# Chapter 3 &mdash; $L^{**} = L^*$: Star is Idempotent

**Concept 12 of the Chapter 3 decomposition:** *Theorem: $L^{**}=L^*$*

The chapter's one theorem, proved by <b>double inclusion</b> using Definition 3 of star.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter3/Concept-Star-Star-Is-Star/Concept-Star-Star-Is-Star.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


**Theorem.** For any language $L$, $L^{**} = L^*$.

The proof runs on **Definition 3**: $L^* = \{x : x \in L^k$ for some $k\}$.

* $L^* \subseteq L^{**}$: if $x \in L^k$, take $m_1 = k$ and every other $m_i = 0$,
  satisfying $k = k+0+\cdots+0$.
* $L^{**} \subseteq L^*$: given $m_1,\ldots,m_k$, choose $k' = \sum m_i$.

The whole proof is **arithmetic on exponents** &mdash; which is why Definition 3 was worth
having.

## 2. Definitions

### The double-inclusion method

In [ ]:
def equal_by_double_inclusion(A, B):
    return lissubset(A,B) and lissubset(B,A)

### Computing $L^{**}$ at a bound

In [ ]:
def star_star(L, n):
    return lstar(lstar(L, n), n)

<!-- nav-strip -->

---

&larr;&nbsp;[Ch3&nbsp;11.&nbsp;The Crux of Homomorphisms](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter3/Concept-Crux-Of-Homomorphisms/Concept-Crux-Of-Homomorphisms.ipynb) &nbsp;&middot;&nbsp; [**Chapter 3** index](https://github.com/ganeshutah/Jove/blob/master/Chapter3/README.md) &nbsp;&middot;&nbsp; [Ch3&nbsp;13.&nbsp;Lexicographic Order, and Why It Fails to Enumerate](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter3/Concept-Lexicographic-Order/Concept-Lexicographic-Order.ipynb)&nbsp;&rarr;

---

## 3. Tests

We cannot compare `lstar(L,n)` with `star_star(L,n)` directly &mdash; the two carry
*different bounds*, so `L**` legitimately reaches longer strings. Instead we check the
**two inclusions of the proof**, each at the bound it actually needs.

In [ ]:
# Direction 1:  L*  is contained in  L**     (take m1 = n, all other m = 0)
for L in [{'ab'}, {'0','1'}, {'a','ab'}, lunit(), lphi()]:
    n = 3
    assert lissubset(lstar(L,n), star_star(L,n)), sorted(L)
    print("L=%-14s  L*_%d  subset of  L**  : True" % (str(sorted(L))[:14], n))

Direction 2: every string of $L^{**}$ built from exponents $m_1..m_k$ lies in
$L^{k'}$ for $k' = \sum m_i$ &mdash; so `star_star(L,n)` fits inside `lstar(L, n*n)`.

In [ ]:
for L in [{'ab'}, {'0','1'}, {'a','ab'}, lunit(), lphi()]:
    n = 3
    assert lissubset(star_star(L,n), lstar(L, n*n)), sorted(L)
    print("L=%-14s  L**  subset of  L*_%d : True" % (str(sorted(L))[:14], n*n))
print()
print("Both inclusions hold everywhere tested -> L** = L*.")

The $\subseteq$ direction that needs $L^0=\{\varepsilon\}$: pad with zero exponents.

In [ ]:
L = {'ab'}
x = 'abab'                       # x is in L^2
print("x =", x, " in L*  ?", x in lstar(L,3))
print("            in L** ?", x in star_star(L,3))
print()
print("Proof: take m1 = 2 and every other m = 0.  2 = 2+0+0+...")
print("Those padding factors contribute nothing because L^0 = {epsilon}.")

The other direction: sum the exponents.

In [ ]:
print("Pick m1=1, m2=2 inside L**  -> total exponent 3 -> x is in L^3")
print("'ababab' in L^3 ?", 'ababab' in lexp(L,3))
assert 'ababab' in lexp(L,3)
print()
print("Both inclusions are just arithmetic on exponents. That is why Definition 3 wins.")

## 4. Exercises


1. Write the proof out using Definition 1 instead. Where does it get unpleasant?
2. Is $(L^*)^+ = L^*$? What about $(L^+)^* $?
3. Chapter 8 uses $(R^*)^* = R^*$ to simplify regular expressions. Find a case where
   it shrinks an expression noticeably.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter3/Concept-Star-Star-Is-Star')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')